In [3]:
# 1. Configuration et Imports
import pandas as pd
import numpy as np
import os

# le fichier est dans 'processed',
INPUT_PATH = "../data/processed/dvf_load.parquet"
OUTPUT_PATH = "../data/processed/dvf_processed.parquet"

# Correction : Utilisation de read_parquet au lieu de read_csv
df = pd.read_parquet(INPUT_PATH)

print(f"Données chargées avec succès : {df.shape[0]} lignes.")

Données chargées avec succès : 1387077 lignes.


In [5]:
# 2. Nettoyage de base (Types et Filtrage Type de bien)
df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")
df["valeur_fonciere"] = pd.to_numeric(df["valeur_fonciere"], errors="coerce")
df["surface_reelle_bati"] = pd.to_numeric(df["surface_reelle_bati"], errors="coerce")
df["surface_terrain"] = df["surface_terrain"].fillna(0)
df["nombre_pieces_principales"] = pd.to_numeric(df["nombre_pieces_principales"], errors="coerce").fillna(1)

df = df[df["nature_mutation"].isin(["Vente", "Vente en l'état futur d'achèvement"])].copy()
df = df[df["type_local"].isin(["Maison", "Appartement"])].copy()

# 3. Regroupement par ID Mutation
# Note : suppression des colonnes DPE/Age qui n'existent pas dans ton nouveau fichier
agg_rules = {
    "surface_reelle_bati": "sum", "nombre_pieces_principales": "sum", "surface_terrain": "sum",
    "valeur_fonciere": "max", "date_mutation": "first", "nature_mutation": "first",
    "type_local": "first", "code_departement": "first", "code_commune": "first",
    "longitude": "first", "latitude": "first"
}

df_clean = df.groupby("id_mutation").agg(agg_rules).reset_index()
df_clean["prix_m2"] = df_clean["valeur_fonciere"] / df_clean["surface_reelle_bati"].replace(0, np.nan)
df_clean = df_clean.dropna(subset=["prix_m2"])

# 4. Nettoyage Géographique
df_clean = df_clean.dropna(subset=["longitude", "latitude"])
df_clean = df_clean[(df_clean["longitude"] != 0) & (df_clean["latitude"] != 0)].copy()

# 5. Nettoyage des surfaces et pièces
df_clean.loc[df_clean["nombre_pieces_principales"] == 0, "nombre_pieces_principales"] = 1
df_clean["surface_par_piece"] = df_clean["surface_reelle_bati"] / df_clean["nombre_pieces_principales"]

df_clean = df_clean[
    (df_clean["surface_reelle_bati"] >= 10) & (df_clean["surface_reelle_bati"] <= 500) &
    (df_clean["surface_par_piece"] >= 7) & (df_clean["surface_par_piece"] <= 80)
].copy()

# 6. Filtrage prix
df_clean = df_clean[
    (df_clean["valeur_fonciere"] >= 20000) & (df_clean["valeur_fonciere"] <= 2000000) &
    (df_clean["prix_m2"] >= 300) & (df_clean["prix_m2"] <= 15000)
].copy()

# 7. Feature Engineering
df_clean["annee"] = df_clean["date_mutation"].dt.year
df_clean["mois"] = df_clean["date_mutation"].dt.month
df_clean["is_maison"] = (df_clean["type_local"] == "Maison").astype(int)
df_clean["is_neuf"] = (df_clean["nature_mutation"] == "Vente en l'état futur d'achèvement").astype(int)

# Calcul ancienneté (base 2025 comme dans tes tests)
df_clean["anciennete_mois"] = (2025 - df_clean["annee"]) * 12 + (6 - df_clean["mois"])

# Nettoyage final avant export
df_clean = df_clean.drop(columns=["nature_mutation", "type_local"])
df_clean.to_parquet("../data/processed/dvf_processed.parquet", index=False)

print("Nettoyage terminé avec succès.")

Nettoyage terminé avec succès.


In [7]:
print("--- 1. Aperçu des données ---")
display(df_clean.head())

print("\n--- 2. Valeurs manquantes (NaN) ---")
print(df_clean.isnull().sum())

print("\n--- 3. Vérification des zéros (valeurs suspectes) ---")
# On compte les 0 dans les colonnes qui ne devraient pas en avoir
zero_cols = ["valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales", "prix_m2"]
for col in zero_cols:
    count = (df_clean[col] == 0).sum()
    print(f"Nombre de zéros dans {col} : {count}")

print("\n--- 4. Statistiques descriptives (Min, Max, Moyenne) ---")
# Transposé pour mieux lire
display(df_clean.describe().T)

print("\n--- 5. Vérification des types ---")
print(df_clean.dtypes)

--- 1. Aperçu des données ---


,id_mutation,surface_reelle_bati,nombre_pieces_principales,surface_terrain,valeur_fonciere,date_mutation,code_departement,code_commune,longitude,latitude,prix_m2,surface_par_piece,annee,mois,is_maison,is_neuf,anciennete_mois
0,2025-1,111.0,5.0,133.0,468000.0,2025-01-07,01,01158,5.907186,46.170782,4216.216216,22.200000,2025,1,1,0,5
1,2025-10,190.0,5.0,860.0,1165000.0,2025-01-09,01,01419,5.981131,46.243443,6131.578947,38.000000,2025,1,1,0,5
3,2025-100000,181.0,9.0,606.0,136500.0,2025-05-19,23,23245,1.807211,46.225608,754.143646,20.111111,2025,5,0,0,1
4,2025-100001,70.0,4.0,52.0,23000.0,2025-05-22,23,23229,2.109319,46.086352,328.571429,17.500000,2025,5,1,0,1
5,2025-100004,114.0,5.0,196.0,68000.0,2025-04-30,23,23173,1.811907,45.948457,596.491228,22.800000,2025,4,1,0,2



--- 2. Valeurs manquantes (NaN) ---
id_mutation                  0
surface_reelle_bati          0
nombre_pieces_principales    0
surface_terrain              0
valeur_fonciere              0
date_mutation                0
code_departement             0
code_commune                 0
longitude                    0
latitude                     0
prix_m2                      0
surface_par_piece            0
annee                        0
mois                         0
is_maison                    0
is_neuf                      0
anciennete_mois              0
dtype: int64

--- 3. Vérification des zéros (valeurs suspectes) ---
Nombre de zéros dans valeur_fonciere : 0
Nombre de zéros dans surface_reelle_bati : 0
Nombre de zéros dans nombre_pieces_principales : 0
Nombre de zéros dans prix_m2 : 0

--- 4. Statistiques descriptives (Min, Max, Moyenne) ---


,count,mean,min,25%,50%,75%,max,std
surface_reelle_bati,310700.0,95.004837,10.0,54.0,79.0,111.0,500.0,67.133449
nombre_pieces_principales,310700.0,4.070222,1.0,2.0,4.0,5.0,56.0,2.557642
surface_terrain,310700.0,520.086044,0.0,0.0,87.0,560.0,1424555.0,3700.554758
valeur_fonciere,310700.0,237945.83961,20000.0,119000.0,186000.0,290000.0,2000000.0,198007.421327
date_mutation,310700,2025-03-21 21:26:01.289990144,2025-01-01 00:00:00,2025-02-12 00:00:00,2025-03-20 00:00:00,2025-04-25 00:00:00,2025-06-30 00:00:00,NaN
longitude,310700.0,2.618762,-63.114998,0.635753,2.441025,4.81768,55.826186,6.333378
latitude,310700.0,46.087561,-21.385665,44.182652,46.832389,48.821715,51.082008,6.268032
prix_m2,310700.0,3016.5885,300.0,1490.566038,2452.830189,3818.495968,15000.0,2227.68566
surface_par_piece,310700.0,23.522237,7.0,19.25,22.25,26.333333,80.0,6.905272
annee,310700.0,2025.0,2025.0,2025.0,2025.0,2025.0,2025.0,0.0



--- 5. Vérification des types ---
id_mutation                          object
surface_reelle_bati                 float64
nombre_pieces_principales           float64
surface_terrain                     float64
valeur_fonciere                     float64
date_mutation                datetime64[ns]
code_departement                     object
code_commune                         object
longitude                           float64
latitude                            float64
prix_m2                             float64
surface_par_piece                   float64
annee                                 int32
mois                                  int32
is_maison                             int64
is_neuf                               int64
anciennete_mois                       int32
dtype: object
